## Experimental Protocol

The objective of this analysis is to evaluate **Ridge Regression** and **Random Forest Regression** for predicting industrial energy consumption while implementing a reproducible preprocessing and cross-validation workflow.

The analysis will be performed as follows:

1. **Target definition**
   The prediction target will be energy consumption, `Usage_kWh`.

2. **Feature selection**
   Numerical and categorical predictors will be included. Initially, all candidate features considered available for prediction will be used.

3. **Potential leakage assessment**
   Particular attention will be given to `CO2(tCO2)`, since CO₂ emissions may be directly or indirectly related to energy consumption. The model will later be evaluated without this feature to investigate its influence on predictive performance.

4. **Cross-validation protocol**
   A consistent cross-validation strategy will be defined and used for all model comparisons so that performance estimates are directly comparable.

5. **Preprocessing pipeline definition**
   Separate scikit-learn pipelines will be constructed for Ridge Regression and Random Forest:

   * Ridge Regression requires scaling of numerical variables and encoding of categorical variables.
   * Random Forest does not require numerical feature scaling, but categorical variables still require appropriate encoding.

6. **Cross-validation of candidate models**
   Ridge Regression and Random Forest will be evaluated using their respective pipelines. Preprocessing will therefore be performed independently within each training fold, reducing the risk of preprocessing-related data leakage.

7. **Model comparison**
   Cross-validation performance and variability across folds will be compared to evaluate predictive accuracy and model stability.

8. **CO₂ sensitivity analysis**
   The Random Forest evaluation will be repeated after removing `CO2(tCO2)` from the feature set while keeping the same validation protocol.

9. **Leakage investigation**
   Performance with and without CO₂ will be compared. A substantial decrease after removing CO₂ would indicate that the feature contains strong predictive information. This alone does not prove data leakage; the origin of the CO₂ variable and whether it would legitimately be available at prediction time must also be investigated.

### Main Objective

The goal is not only to identify the best-performing model, but to ensure that the estimated performance reflects a **realistic, reproducible, and leakage-safe machine-learning workflow**.


In [7]:
#Step 1 - Target definition
import pandas as pd
df = pd.read_csv("Steel_industry_data.csv")

In [8]:
y = df["Usage_kWh"]
df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


In [22]:
#Step 2 & 3 - Feature selection and definition + train/test split
features_all_numerical = [
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "CO2(tCO2)",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
    "NSM"
]

features_noCO2_numerical = [
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
    "NSM"
]

features_all_categorical = [
    "WeekStatus",
    "Day_of_week",
    "Load_Type",
]

features_all = features_all_numerical + features_all_categorical
features_noCO2 = features_noCO2_numerical + features_all_categorical

In [23]:
print(features_all)

['Lagging_Current_Reactive.Power_kVarh', 'Leading_Current_Reactive_Power_kVarh', 'CO2(tCO2)', 'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor', 'NSM', 'WeekStatus', 'Day_of_week', 'Load_Type']


In [40]:
X_all = df[features_all]

from sklearn.model_selection import train_test_split

X_all_train, X_all_test, y_all_train, y_all_test = train_test_split(
    X_all,
    y,
    test_size = 0.2,
    random_state = 42
)

In [25]:
#Step 4 - Cross validation protocol
from sklearn.model_selection import KFold

CV = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

In [28]:
#Step 5 - Preprocessor + Pipeline definitions
#Ridge (scaling + encoding)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor_ridge = ColumnTransformer ([
    (
        "num",
        StandardScaler(),
        features_all_numerical
    ),
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore"),
        features_all_categorical
    )
])

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

pipeline_ridge = Pipeline([
    ("preprocessing", preprocessor_ridge),
    ("model", Ridge(alpha=1.0))
])

#RF (encoding only)

from sklearn.ensemble import RandomForestRegressor

preprocessor_rf = ColumnTransformer ([
    (
        "num",
        "passthrough",
        features_all_numerical
    ),
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore"),
        features_all_categorical
    )
])

pipeline_rf = Pipeline([
    ("preprocessing", preprocessor_rf),
    ("model", 
     RandomForestRegressor(
             n_estimators = 300,
             random_state = 42,
             n_jobs = -1
         )
    )
])

In [43]:
#Step 6 - Cross validation of candidate models
from sklearn.model_selection import cross_validate
scores_ridge = cross_validate(
    pipeline_ridge,
    X_all_train,
    y_all_train,
    cv=CV,
    scoring = {
        "r2":"r2",
        "mae":"neg_mean_absolute_error"
    },
    return_train_score=True
)
print("test r2: ", scores_ridge["test_r2"].mean())
print("train r2: ",scores_ridge["train_r2"].mean())
print("test mae: ",-scores_ridge["test_mae"].mean())
print("train mae: ",-scores_ridge["train_mae"].mean())

test r2:  0.9794029702432605
train r2:  0.9794416671025188
test mae:  2.5971451053651484
train mae:  2.5954370873777095


In [44]:
scores_rf = cross_validate(
    pipeline_rf,
    X_all_train,
    y_all_train,
    cv=CV,
    scoring = {
        "r2":"r2",
        "mae":"neg_mean_absolute_error"
    },
    return_train_score=True
)
print("test r2: ", scores_rf["test_r2"].mean())
print("train r2: ",scores_rf["train_r2"].mean())
print("test mae: ",-scores_rf["test_mae"].mean())
print("train mae: ",-scores_rf["train_mae"].mean())

test r2:  0.9988704980493711
train r2:  0.9998497180985858
test mae:  0.3633785422858046
train mae:  0.134028640600344


In [50]:
#Step 8 - CO2 sensitivity
X_noCO2 = df[features_noCO2]

X_noCO2_train = X_all_train.drop(
    columns=["CO2(tCO2)"]
)

X_noCO2_test = X_all_test.drop(
    columns=["CO2(tCO2)"]
)

preprocessor_ridge_noCO2 = ColumnTransformer ([
    (
        "num",
        StandardScaler(),
        features_noCO2_numerical
    ),
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore"),
        features_all_categorical
    )
])

pipeline_ridge_noCO2 = Pipeline([
    ("preprocessing", preprocessor_ridge_noCO2),
    ("model", Ridge(alpha=1.0))
])

preprocessor_rf_noCO2 = ColumnTransformer ([
    (
        "num",
        "passthrough",
        features_noCO2_numerical
    ),
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore"),
        features_all_categorical
    )
])

pipeline_rf_noCO2 = Pipeline([
    ("preprocessing", preprocessor_rf_noCO2),
    ("model", 
     RandomForestRegressor(
             n_estimators = 300,
             random_state = 42,
             n_jobs = -1
         )
    )
])

scores_ridge_noCO2 = cross_validate(
    pipeline_ridge_noCO2,
    X_noCO2_train,
    y_all_train,
    cv=CV,
    scoring = {
        "r2":"r2",
        "mae":"neg_mean_absolute_error"
    },
    return_train_score=True
)
print("CV r2 no CO2: ", scores_ridge_noCO2["test_r2"].mean())
print("train r2 no CO2: ",scores_ridge_noCO2["train_r2"].mean())
print("test mae no CO2: ",-scores_ridge_noCO2["test_mae"].mean())
print("train mae no CO2: ",-scores_ridge_noCO2["train_mae"].mean())

scores_rf_noCO2 = cross_validate(
    pipeline_rf_noCO2,
    X_noCO2_train,
    y_all_train,
    cv=CV,
    scoring = {
        "r2":"r2",
        "mae":"neg_mean_absolute_error"
    },
    return_train_score=True
)
print("CV r2 no CO2: ", scores_rf_noCO2["test_r2"].mean())
print("train r2 no CO2: ",scores_rf_noCO2["train_r2"].mean())
print("test mae no CO2: ",-scores_rf_noCO2["test_mae"].mean())
print("train mae no CO2: ",-scores_rf_noCO2["train_mae"].mean())

CV r2 no CO2:  0.9174150576872815
train r2 no CO2:  0.9175504526381506
test mae no CO2:  6.878347504898026
train mae no CO2:  6.875085138259422
CV r2 no CO2:  0.9993886912991693
train r2 no CO2:  0.9999193156890438
test mae no CO2:  0.25452052783539747
train mae no CO2:  0.09513614816112162


In [53]:
#Step 9 - Leakage investigation

results = pd.DataFrame({
    "Model":[
        "Ridge",
        "Random Forest",
        "Ridge",
        "Random Forest"
    ],
    "Features":[
        "ALL",
        "ALL",
        "No CO2",
        "No CO2"
    ],
    "Train R2":[
        scores_ridge["train_r2"].mean(),
        scores_rf["train_r2"].mean(),
        scores_ridge_noCO2["train_r2"].mean(),
        scores_rf_noCO2["train_r2"].mean()
    ],
    "CV R2":[
        scores_ridge["test_r2"].mean(),
        scores_rf["test_r2"].mean(),
        scores_ridge_noCO2["test_r2"].mean(),
        scores_rf_noCO2["test_r2"].mean()
    ],
    "CV R2 STD":[
        scores_ridge["test_r2"].std(),
        scores_rf["test_r2"].std(),
        scores_ridge_noCO2["test_r2"].std(),
        scores_rf_noCO2["test_r2"].std()
    ],
    "Train MAE":[
        scores_ridge["train_mae"].mean(),
        scores_rf["train_mae"].mean(),
        scores_ridge_noCO2["train_mae"].mean(),
        scores_rf_noCO2["train_mae"].mean()
    ],
    "CV MAE":[
        scores_ridge["test_mae"].mean(),
        scores_rf["test_mae"].mean(),
        scores_ridge_noCO2["test_mae"].mean(),
        scores_rf_noCO2["test_mae"].mean()
    ],
    "CV MAE STD":[
    scores_ridge["test_mae"].std(),
    scores_rf["test_mae"].std(),
    scores_ridge_noCO2["test_mae"].std(),
    scores_rf_noCO2["test_mae"].std()
    ]
})

In [55]:
display(results)

,Model,Features,Train R2,CV R2,CV R2 STD,Train MAE,CV MAE,CV MAE STD
0,Ridge,ALL,0.979442,0.979403,0.001517,-2.595437,-2.597145,0.031054
1,Random Forest,ALL,0.999850,0.998870,0.000085,-0.134029,-0.363379,0.007443
2,Ridge,No CO2,0.917550,0.917415,0.003034,-6.875085,-6.878348,0.060605
3,Random Forest,No CO2,0.999919,0.999389,0.000053,-0.095136,-0.254521,0.004863


## Conclusion

A leakage-safe preprocessing and cross-validation workflow was implemented to compare Ridge Regression and Random Forest Regression for predicting industrial energy consumption.

Random Forest substantially outperformed Ridge Regression. Using all selected features, Random Forest achieved a cross-validation R² of approximately **0.999** and a CV MAE of approximately **0.36 kWh**, compared with approximately **0.979 R²** and **2.60 kWh MAE** for Ridge Regression.

To investigate whether the very high Random Forest performance could be caused by the `CO2(tCO2)` feature, the models were evaluated again after removing CO₂ while keeping the validation protocol unchanged.

Removing CO₂ substantially reduced Ridge performance, indicating that CO₂ contains strong predictive information for the linear model. However, Random Forest performance did not decrease. Its CV R² remained approximately **0.999**, while CV MAE slightly improved.

Therefore, the exceptionally strong Random Forest performance cannot be explained by the CO₂ feature alone. Additional investigation is required to determine whether other predictors, particularly simultaneously measured electrical variables such as reactive power and power factor, contain enough information to reconstruct energy consumption almost directly.

This experiment also demonstrated the importance of using scikit-learn pipelines and cross-validation to ensure that preprocessing is learned only from training observations within each fold, reducing the risk of preprocessing-related data leakage.
